In [1]:
import pandas as pd
df = pd.read_csv('data/Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [2]:
from sklearn.model_selection import train_test_split
y = (df['Churn'] == 'Yes').astype(int)
X = df.drop(columns=['Churn', 'customerID'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [3]:
print(X.shape)
print(X_train.shape, X_test.shape)
print(f'Churn train: {y_train.mean():.3f}')
print(f'Churn test: {y_test.mean():.3f}')

(7043, 19)
(5634, 19) (1409, 19)
Churn train: 0.265
Churn test: 0.265


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_cols = ['tenure','MonthlyCharges','TotalCharges']
categorical_cols = X_train.select_dtypes(include=['object','string']).columns.tolist()

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant',fill_value=0)),
    ('scaler',StandardScaler()),
])
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
       ('num', numeric_transformer,numeric_cols),
       ('cat',categorical_transformer,categorical_cols),
    ],
    remainder='passthrough',
)
X_train_prep = preprocessor.fit_transform(X_train)
print(X_train_prep.shape)

(5634, 45)


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

logreg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42)),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'roc_auc':'roc_auc',
    'recall':'recall',
    'precision':'precision',
    'pr_auc':'average_precision',
}
results = cross_validate(logreg_pipeline, X_train, y_train, cv=cv, scoring=scoring)
for name in scoring:
    scores = results[f'test_{name}']
    print(f'{name}: {scores.mean():.3f} +/- {scores.std():.3f}')

roc_auc: 0.846 +/- 0.013
recall: 0.544 +/- 0.041
precision: 0.654 +/- 0.027
pr_auc: 0.662 +/- 0.019
